# CPSC 483 — Day 5 Class Work #1
## SVM: Classifying Iris setosa vs. virginica

- Use **only Petal Length and Petal Width**
- Ignore instances of **versicolor**
- Report the **equation of the decision boundary**
- Report **how many support vectors** are needed

## Setup

In [ ]:
import sys
assert sys.version_info >= (3, 10)

from packaging.version import Version
import sklearn
assert Version(sklearn.__version__) >= Version("1.6.1")

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Helper — textbook `plot_svc_decision_boundary`

From *Appendix C – Support Vector Machines* (Géron 2025).  
Draws the decision boundary (`w·x + b = 0`) and both margin gutters (`= ±1`),  
then shades the support vectors.

In [ ]:
import numpy as np

def plot_svc_decision_boundary(svm_clf, xmin, xmax):
    w = svm_clf.coef_[0]
    b = svm_clf.intercept_[0]

    # At the decision boundary, w0*x0 + w1*x1 + b = 0
    # => x1 = -w0/w1 * x0 - b/w1
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0] / w[1] * x0 - b / w[1]

    margin = 1 / w[1]
    gutter_up   = decision_boundary + margin
    gutter_down = decision_boundary - margin
    svs = svm_clf.support_vectors_

    plt.plot(x0, decision_boundary, "k-",  linewidth=2, zorder=-2)
    plt.plot(x0, gutter_up,         "k--", linewidth=2, zorder=-2)
    plt.plot(x0, gutter_down,       "k--", linewidth=2, zorder=-2)
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#AAA', zorder=-1)

## 1. Load data — petal features only, keep setosa and virginica

In [ ]:
from sklearn import datasets

iris = datasets.load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris.target   # 0 = setosa, 1 = versicolor, 2 = virginica

# Keep setosa (0) and virginica (2) — ignore versicolor (1)
setosa_or_virginica = (y == 0) | (y == 2)
X = X[setosa_or_virginica]
y = y[setosa_or_virginica]

print("Classes:", sorted(y.unique()), "→",
      [iris.target_names[c] for c in sorted(y.unique())])
print("Shape: ", X.shape)

## 2. Train the SVM

`C=1e100` makes the penalty for any margin violation astronomically large —  
effectively a **hard margin** classifier.  
Setosa and virginica are perfectly separable in petal space, so a hard-margin solution exists.

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel="linear", C=1e100)
svm.fit(X, y)

## 3. Decision boundary equation

For a linear SVM, the boundary is the hyperplane where the decision function equals zero:

$$w_0 \cdot \text{petal\_length} \;+\; w_1 \cdot \text{petal\_width} \;+\; b \;=\; 0$$

Solving for petal width gives the line we can plot:

$$\text{petal\_width} = -\frac{w_0}{w_1} \cdot \text{petal\_length} - \frac{b}{w_1}$$

In [ ]:
w = svm.coef_[0]       # [w0, w1]
b = svm.intercept_[0]  # bias term

print("Decision boundary (implicit form):")
print(f"  ({w[0]:.4f}) · petal_length  +  ({w[1]:.4f}) · petal_width  +  ({b:.4f})  =  0")
print()
print("Decision boundary (slope-intercept form):")
print(f"  petal_width  =  ({-w[0]/w[1]:.4f}) · petal_length  +  ({-b/w[1]:.4f})")

## 4. Support vectors

In [ ]:
print(f"Total support vectors : {len(svm.support_vectors_)}")
print(f"Per class             : {svm.n_support_}  "
      f"({iris.target_names[0]} / {iris.target_names[2]})")
print()
print("Coordinates (petal_length, petal_width):")
for i, sv in enumerate(svm.support_vectors_):
    print(f"  SV {i+1}: ({sv[0]:.1f},  {sv[1]:.1f})")

## 5. Visualization

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "yo", label="Iris setosa")
plt.plot(X[:, 0][y == 2], X[:, 1][y == 2], "bs", label="Iris virginica")

plot_svc_decision_boundary(svm, xmin=0.5, xmax=7.5)

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Linear SVM — Setosa vs. Virginica")
plt.legend(loc="upper left")
plt.axis([0.5, 7.5, -0.1, 2.8])
plt.gca().set_aspect("equal")
plt.grid()
plt.show()

## Summary — answers to the classwork questions

In [ ]:
print("=" * 58)
print("CLASS WORK #1 — ANSWERS")
print("=" * 58)

print("\nQ1 — Decision boundary equation:")
print(f"  ({w[0]:.4f})·petal_length + ({w[1]:.4f})·petal_width + ({b:.4f}) = 0")
print(f"  → petal_width = ({-w[0]/w[1]:.4f})·petal_length + ({-b/w[1]:.4f})")

print(f"\nQ2 — Support vectors needed: {len(svm.support_vectors_)}")
print(f"  {svm.n_support_[0]} from setosa, {svm.n_support_[1]} from virginica")
print("  (Only the points that sit on the margin edges matter;")
print("   all other training points are irrelevant to the boundary.)")
print("=" * 58)